In [125]:
import pandas as pd
import geopandas as gpd
import duckdb

In [193]:
#connect to duckdb
con = duckdb.connect('../database/spatial-db.db', read_only=False)

In [194]:
#install the spatial extension
con.install_extension("spatial")
con.load_extension("spatial")

# State

In [199]:
states = gpd.read_file('data/00_state/state.shp')

In [200]:
states.head()

,GEOID,state_name,ppl_densit,transit_to,walk_to_wo,c_lat,c_lon,neighbors_,geometry
0,1,Alabama,99.26968,0.003771,0.011017,32.756880,-86.844521,"['Tennessee', 'Florida', 'Mississippi', 'Georg...","POLYGON ((-85.12733 31.76256, -85.12753 31.762..."
1,4,Arizona,63.10557,0.014370,0.016875,34.293141,-111.664444,"['Utah', 'none', 'California', 'New Mexico']","POLYGON ((-110.75069 37.00301, -110.74193 37.0..."
2,5,Arkansas,58.05936,0.003333,0.015007,34.899917,-92.438374,"['Missouri', 'Louisiana', 'Oklahoma', 'Mississ...","POLYGON ((-90.95577 34.11871, -90.95451 34.117..."
3,6,California,252.51050,0.038160,0.023834,37.215333,-119.663871,"['Oregon', 'none', 'none', 'Nevada']","MULTIPOLYGON (((-119.99987 41.18397, -119.9998..."
4,8,Colorado,55.68269,0.022252,0.026464,38.998532,-105.547821,"['Wyoming', 'New Mexico', 'Utah', 'Nebraska']","POLYGON ((-105.15504 36.99526, -105.15543 36.9..."


In [130]:
states = states.copy() 
states["centroid"] = states["geometry"].centroid

/var/folders/zm/x5dzdlrs5mlbxrm7z5g6p05r0000gn/T/ipykernel_15369/695611934.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  states["centroid"] = states["geometry"].centroid


In [131]:
def find_neighbors(row, gdf):
    # "touches" checks if boundaries touch (i.e., share at least one point)
    # We filter out the state itself by comparing indices
    return gdf[gdf.geometry.touches(row.geometry)].index

# Collect neighbor indices for each state
neighbors_dict = {}
for idx, row in states.iterrows():
    neighbors_dict[idx] = find_neighbors(row, states)

# Store the neighbor indices in a column for convenience
states["neighbor_indices"] = states.index.map(neighbors_dict)

In [132]:
def get_direction(centroid1, centroid2):
    """
    Classify the direction of centroid2 relative to centroid1
    as one of: 'north', 'south', 'west', 'east'.
    """
    dx = centroid2.x - centroid1.x
    dy = centroid2.y - centroid1.y
    
    # Compare absolute differences to decide axis of dominance
    if abs(dy) > abs(dx):
        return "north" if dy > 0 else "south"
    else:
        return "east" if dx > 0 else "west"

In [133]:
directions = ["north", "south", "west", "east"]

results = []
for idx, row in states.iterrows():
    c1 = row["centroid"]
    neighbor_idxs = row["neighbor_indices"]
    
    # Track best neighbor in each direction as (neighbor_name, distance_so_far)
    best_in_dir = {d: (None, float("inf")) for d in directions}
    
    # Check all neighbors
    for n_idx in neighbor_idxs:
        neighbor_row = states.loc[n_idx]
        c2 = neighbor_row["centroid"]
        d = get_direction(c1, c2)
        
        # Compute actual distance (using centroid points)
        dist = c1.distance(c2)
        
        # If it's smaller than the best we have so far, update
        if dist < best_in_dir[d][1]:
            best_in_dir[d] = (neighbor_row["state_name"], dist)
    
    # Build final list of 4 directions: [north, south, west, east]
    final_list = []
    for d in directions:
        neighbor_name, _ = best_in_dir[d]
        if neighbor_name is None:
            final_list.append("none")
        else:
            final_list.append(neighbor_name)
    
    results.append(final_list)

states["neighbors_cardinal"] = results

In [134]:
states.loc[states['state_name'] == 'Connecticut', 'neighbors_cardinal']

5    [none, none, New York, Rhode Island]
Name: neighbors_cardinal, dtype: object

In [135]:
states.at[5, 'neighbors_cardinal'] = ['Massachusetts', 'New York', 'New York', 'Rhode Island']

In [136]:
states.loc[states['state_name'] == 'Pennsylvania', 'neighbors_cardinal']

36    [none, Maryland, West Virginia, New York]
Name: neighbors_cardinal, dtype: object

In [137]:
states.at[36, 'neighbors_cardinal'] = ['New York', 'Maryland', 'Ohio', 'New Jersey']

In [138]:
states.loc[states['state_name'] == 'Virginia', 'neighbors_cardinal']

44    [none, North Carolina, West Virginia, District...
Name: neighbors_cardinal, dtype: object

In [139]:
states.at[44, 'neighbors_cardinal'] = ['Maryland', 'North Carolina', 'West Virginia', 'none']

In [140]:
states.loc[states['state_name'] == 'West Virginia', 'neighbors_cardinal']

46    [none, none, Ohio, Virginia]
Name: neighbors_cardinal, dtype: object

In [141]:
states.at[46, 'neighbors_cardinal'] = ['Pennsylvania', 'Virginia', 'Ohio', 'Virginia']

In [142]:
states.loc[states['state_name'] == 'District of Columbia', 'neighbors_']

7    ['Maryland', 'Virginia', 'Virginia', 'Maryland']
Name: neighbors_, dtype: object

In [143]:
states.at[7, 'neighbors_cardinal'] = ['Maryland', 'Virginia', 'Virginia', 'Maryland']

In [144]:
states.loc[states['state_name'] == 'South Carolina', 'neighbors_cardinal']

38    [none, none, Georgia, North Carolina]
Name: neighbors_cardinal, dtype: object

In [145]:
states.at[38, 'neighbors_cardinal'] = ['North Carolina', 'Georgia', 'Georgia', 'none']

In [146]:
states.loc[states['state_name'] == 'North Carolina', 'neighbors_cardinal']

31    [Virginia, none, South Carolina, none]
Name: neighbors_cardinal, dtype: object

In [147]:
states.at[31, 'neighbors_cardinal'] = ['Virginia', 'South Carolina', 'Tennessee', 'none']

In [149]:
states.loc[states['state_name'] == 'Maryland', 'neighbors_cardinal']

18    [Pennsylvania, none, District of Columbia, Del...
Name: neighbors_cardinal, dtype: object

In [150]:
states.at[18, 'neighbors_cardinal'] = ['Pennsylvania', 'Virginia', 'District of Columbia', 'none']

In [152]:
states.head()

,GEOID,state_name,ppl_densit,transit_to,walk_to_wo,c_lat,c_lon,neighbors_,geometry,centroid,neighbor_indices,neighbors_cardinal
0,1,Alabama,99.26968,0.003771,0.011017,32.756880,-86.844521,"['Tennessee', 'Florida', 'Mississippi', 'Georg...","POLYGON ((-85.12733 31.76256, -85.12753 31.762...",POINT (-86.84452 32.75688),"Index([8, 9, 22, 40], dtype='int64')","[Tennessee, Florida, Mississippi, Georgia]"
1,4,Arizona,63.10557,0.014370,0.016875,34.293141,-111.664444,"['Utah', 'none', 'California', 'New Mexico']","POLYGON ((-110.75069 37.00301, -110.74193 37.0...",POINT (-111.66444 34.29314),"Index([3, 4, 26, 29, 42], dtype='int64')","[Utah, none, California, New Mexico]"
2,5,Arkansas,58.05936,0.003333,0.015007,34.899917,-92.438374,"['Missouri', 'Louisiana', 'Oklahoma', 'Mississ...","POLYGON ((-90.95577 34.11871, -90.95451 34.117...",POINT (-92.43837 34.89992),"Index([16, 22, 23, 34, 40, 41], dtype='int64')","[Missouri, Louisiana, Oklahoma, Mississippi]"
3,6,California,252.51050,0.038160,0.023834,37.215333,-119.663871,"['Oregon', 'none', 'none', 'Nevada']","MULTIPOLYGON (((-119.99987 41.18397, -119.9998...",POINT (-119.66387 37.21533),"Index([1, 26, 35], dtype='int64')","[Oregon, none, none, Nevada]"
4,8,Colorado,55.68269,0.022252,0.026464,38.998532,-105.547821,"['Wyoming', 'New Mexico', 'Utah', 'Nebraska']","POLYGON ((-105.15504 36.99526, -105.15543 36.9...",POINT (-105.54782 38.99853),"Index([1, 14, 25, 29, 34, 42, 48], dtype='int64')","[Wyoming, New Mexico, Utah, Nebraska]"


In [153]:
states.drop(columns=['centroid', 'neighbor_indices', 'neighbors_'], inplace=True)

In [154]:
states = states[['GEOID','state_name', 'ppl_densit','transit_to','walk_to_wo','c_lat','c_lon','neighbors_cardinal', 'geometry']]

In [155]:
states.head()

,GEOID,state_name,ppl_densit,transit_to,walk_to_wo,c_lat,c_lon,neighbors_cardinal,geometry
0,1,Alabama,99.26968,0.003771,0.011017,32.756880,-86.844521,"[Tennessee, Florida, Mississippi, Georgia]","POLYGON ((-85.12733 31.76256, -85.12753 31.762..."
1,4,Arizona,63.10557,0.014370,0.016875,34.293141,-111.664444,"[Utah, none, California, New Mexico]","POLYGON ((-110.75069 37.00301, -110.74193 37.0..."
2,5,Arkansas,58.05936,0.003333,0.015007,34.899917,-92.438374,"[Missouri, Louisiana, Oklahoma, Mississippi]","POLYGON ((-90.95577 34.11871, -90.95451 34.117..."
3,6,California,252.51050,0.038160,0.023834,37.215333,-119.663871,"[Oregon, none, none, Nevada]","MULTIPOLYGON (((-119.99987 41.18397, -119.9998..."
4,8,Colorado,55.68269,0.022252,0.026464,38.998532,-105.547821,"[Wyoming, New Mexico, Utah, Nebraska]","POLYGON ((-105.15504 36.99526, -105.15543 36.9..."


In [156]:
# gdf = gpd.read_file('data/00_state/state.shp')
# #sort by GEOID
# gdf = gdf.sort_values(by='GEOID')
# # reset the index
# gdf = gdf.reset_index(drop=True)
# save to shapefile
states.to_file('data/00_state/state.shp')

/var/folders/zm/x5dzdlrs5mlbxrm7z5g6p05r0000gn/T/ipykernel_15369/2470533202.py:7: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  states.to_file('data/00_state/state.shp')
/Users/chuli/opt/anaconda3/envs/spatial-db/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'neighbors_cardinal' to 'neighbors_'
  ogr_write(


In [201]:
#read the shapefile into duckdb
con.sql("SELECT * FROM ST_Read('data/00_state/state.shp')")

┌───────┬──────────────────────┬────────────┬───────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [202]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS state")

In [203]:
#create a table for the state shapefile
con.sql("CREATE TABLE state AS SELECT * FROM ST_Read('data/00_state/state.shp')")

In [204]:
#check if the table was created
con.table('state')

┌───────┬──────────────────────┬────────────┬───────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [205]:
#get all the columns in the table
con.table('state').columns

['GEOID',
 'state_name',
 'ppl_densit',
 'transit_to',
 'walk_to_wo',
 'c_lat',
 'c_lon',
 'neighbors_',
 'geom']

In [206]:
con.close()

# County

In [180]:
county = gpd.read_file('data/01_county/full_county.shp')

In [165]:
county

,GEOID,ppl_densit,transit_to,walk_to_wo,state_name,county_nam,c_lat,c_lon,geometry
0,31039,15.775040,0.000000,0.017324,Nebraska,Cuming,41.916404,-96.787401,"POLYGON ((-96.55516 41.91587, -96.55515 41.914..."
1,53069,17.023660,0.002581,0.041935,Washington,Wahkiakum,46.291134,-123.433470,"POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35011,0.729626,0.000000,0.056787,New Mexico,De Baca,34.342414,-104.411958,"POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31109,384.524800,0.009055,0.030294,Nebraska,Lancaster,40.784174,-96.687756,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31129,7.114601,0.000000,0.045455,Nebraska,Nuckolls,40.176380,-98.047185,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."
...,...,...,...,...,...,...,...,...,...
3104,13123,73.950830,0.002572,0.008928,Georgia,Gilmer,34.691178,-84.455627,"POLYGON ((-84.30237 34.57832, -84.30329 34.577..."
3105,27135,9.148537,0.001804,0.025258,Minnesota,Roseau,48.775147,-95.810806,"POLYGON ((-95.25857 48.88666, -95.25707 48.885..."
3106,28089,152.944400,0.000753,0.005686,Mississippi,Madison,32.634710,-90.033714,"POLYGON ((-90.14883 32.40026, -90.1489 32.4001..."
3107,48227,38.286330,0.000000,0.010094,Texas,Howard,32.306169,-101.435505,"POLYGON ((-101.18138 32.21252, -101.18138 32.2..."


In [167]:
# Make a copy to avoid mutating the original:
counties = county.copy()

# ------------------------------------------------------------------------
# 2. Compute centroid of each county polygon
# ------------------------------------------------------------------------
counties["centroid"] = counties.geometry.centroid

# ------------------------------------------------------------------------
# 3. Build the spatial index
# ------------------------------------------------------------------------
counties_sindex = counties.sindex  # uses rtree or pygeos-based index

# ------------------------------------------------------------------------
# 4. Define a function to get all neighbors in the same state
#    whose geometry "touches" the boundary of the given county
# ------------------------------------------------------------------------
def find_neighbors_in_state(row, gdf, sindex):
    """
    For a given county row, find all counties in the same state
    whose polygon 'touches' this county's polygon.
    """
    # 1) Filter by bounding box + 'touches' via the new .query(..., predicate="touches")
    #    This returns an index of candidate geometry that touches row's geometry
    possible_idxs = sindex.query(row.geometry, predicate="touches")
    
    # 2) Narrow to same state and exclude the row's own index
    possible_neighbors = gdf.loc[possible_idxs]
    neighbors = possible_neighbors[
        (possible_neighbors["state_name"] == row["state_name"]) &
        (possible_neighbors.index != row.name)
    ]
    return neighbors.index

# ------------------------------------------------------------------------
# 5. Create a dictionary mapping each county index -> list of neighbor indices
# ------------------------------------------------------------------------
neighbor_dict = {}
for idx, row in counties.iterrows():
    neighbor_dict[idx] = find_neighbors_in_state(row, counties, counties_sindex)

# Optional: store the neighbor indices in a column
counties["neighbor_indices"] = counties.index.map(neighbor_dict)

# ------------------------------------------------------------------------
# 6. Define a helper to classify direction: north, south, west, or east
# ------------------------------------------------------------------------
def get_direction(centroid1, centroid2):
    """
    Return 'north', 'south', 'west', or 'east' based on which axis
    has the larger difference. (Diagonal counties get lumped into
    whichever axis is dominant.)
    """
    dx = centroid2.x - centroid1.x
    dy = centroid2.y - centroid1.y
    if abs(dy) > abs(dx):
        return "north" if dy > 0 else "south"
    else:
        return "east" if dx > 0 else "west"

# We'll keep a fixed order of directions:
directions = ["north", "south", "west", "east"]

# ------------------------------------------------------------------------
# 7. For each county, pick the closest neighbor in each of the 4 directions
# ------------------------------------------------------------------------
all_neighbors_cardinal = []
for idx, row in counties.iterrows():
    this_centroid = row["centroid"]
    neighbor_idxs = row["neighbor_indices"]
    
    # Track best neighbor for each direction as (name, distance)
    best_in_dir = {d: (None, float("inf")) for d in directions}
    
    # Check each neighbor's centroid
    for n_idx in neighbor_idxs:
        neighbor_row = counties.loc[n_idx]
        nb_centroid = neighbor_row["centroid"]
        
        d = get_direction(this_centroid, nb_centroid)
        dist = this_centroid.distance(nb_centroid)
        
        # Update if this neighbor is closer in that direction
        if dist < best_in_dir[d][1]:
            best_in_dir[d] = (neighbor_row["county_nam"], dist)
    
    # Build a final 4-element list in order [north, south, west, east]
    final_list = []
    for d in directions:
        neighbor_name, _ = best_in_dir[d]
        final_list.append(neighbor_name if neighbor_name else "none")
    
    all_neighbors_cardinal.append(final_list)

# ------------------------------------------------------------------------
# 8. Store the 4-neighbor list in a new column
# ------------------------------------------------------------------------
counties["neighbors_cardinal"] = all_neighbors_cardinal

# Done! Now each row in 'counties' has an array like:
#   ["County A", "County B", "none", "County C"]
# representing the nearest neighbor to the north, south, west, and east
# (if any) within the same state.


/var/folders/zm/x5dzdlrs5mlbxrm7z5g6p05r0000gn/T/ipykernel_15369/10102875.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  counties["centroid"] = counties.geometry.centroid


In [175]:
counties.head()

,GEOID,ppl_densit,transit_to,walk_to_wo,state_name,county_nam,c_lat,c_lon,geometry,centroid,neighbor_indices,neighbors_cardinal
0,31039,15.775040,0.000000,0.017324,Nebraska,Cuming,41.916404,-96.787401,"POLYGON ((-96.55516 41.91587, -96.55515 41.914...",POINT (-96.7874 41.9164),"Index([3065, 2123, 1746, 1927, 1150, 883], dty...","[none, Dodge, Stanton, Thurston]"
1,53069,17.023660,0.002581,0.041935,Washington,Wahkiakum,46.291134,-123.433470,"POLYGON ((-123.72755 46.2645, -123.72756 46.26...",POINT (-123.43347 46.29113),"Index([2212, 1956, 168], dtype='int64')","[none, none, Pacific, Cowlitz]"
2,35011,0.729626,0.000000,0.056787,New Mexico,De Baca,34.342414,-104.411958,"POLYGON ((-104.89337 34.08894, -104.89337 34.0...",POINT (-104.41196 34.34241),"Index([2446, 1763, 1608, 1862, 2542], dtype='i...","[Guadalupe, Chaves, Lincoln, Roosevelt]"
3,31109,384.524800,0.009055,0.030294,Nebraska,Lancaster,40.784174,-96.687756,"POLYGON ((-96.68493 40.5233, -96.69219 40.5231...",POINT (-96.68776 40.78417),"Index([1327, 1395, 329, 533, 2330, 739, 1829, ...","[Saunders, Gage, Seward, Cass]"
4,31129,7.114601,0.000000,0.045455,Nebraska,Nuckolls,40.176380,-98.047185,"POLYGON ((-98.2737 40.1184, -98.27374 40.1224,...",POINT (-98.04718 40.17638),"Index([2788, 80, 2385, 2287], dtype='int64')","[Clay, none, Webster, Thayer]"


In [176]:
counties = counties[['GEOID','ppl_densit','transit_to','walk_to_wo','state_name','county_nam','c_lat','c_lon','neighbors_cardinal', 'geometry']]

In [187]:
#save to shapefile
counties.to_file('data/01_county/full_county.shp')

/var/folders/zm/x5dzdlrs5mlbxrm7z5g6p05r0000gn/T/ipykernel_15369/2161584900.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  counties.to_file('data/01_county/full_county.shp')
/Users/chuli/opt/anaconda3/envs/spatial-db/lib/python3.10/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'neighbors_cardinal' to 'neighbors_'
  ogr_write(


In [197]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS county")

In [198]:
#create a table for the county shapefile
con.sql("CREATE TABLE county AS SELECT * FROM ST_Read('data/01_county/full_county.shp')")
#check if the table was created
con.table('county')

┌───────┬────────────┬───────────────────┬───────────────────┬───────────────┬─────────────┬────────────────────┬─────────────────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [191]:
con.close()